In [ ]:
### 1 Giriş

Adaboost, zayıf öğrencileri bir araya getirerek güçlü bir model oluşturmayı amaçlayan bir **boosting** algoritmasıdır. AdaBoost, genellikle karar ağaçlarının decision stump (derinliği 1 olan karar ağaçları) versiyonlarını kullanır ve bu modellerin hatalarına göre ağırlıklandırma yapar.

### 2 Temel Prensipler

- AdaBoost şu adımları takip eder:
	- Verinin tüm noktalarına eşit ağırlıkla başlanır.
	- İlk zayıf model (stump eğitilir) eğitilir.
	- Yanlış sınıflandırılan örneklerin ağırlıkları arttırılır, doğru sınıflandırılan örneklerin ağırlıkları azaltılır.
	- Ağırlıklar normalleştirilir ve yeni örnek seçimi için **bin** sistemi kurulur.
	- Süreç belirlenen sayıda iterasyon ile tekrar eder.
### 3 AdaBoost Fonksiyonu

![[Pasted image 20260519235913.png]]

- Her gözleme eşit ağırlık verir.
- Zayıf bir model kurar.
- Yanlış tahmin edilen gözlemlerin ağırlığını artırır.
- Doğru tahmin edilenlerin ağırlığını azaltır.
- Yeni model daha çok zor gözlemlere odaklanır.
- Tüm modeller ağırlıklı şekilde birleştirilir.

![[Pasted image 20260520000341.png]]

Örnek algoritma mantığı:

GPA -- Interview Score -- Approval

| GPA  | Interview Score | Approval (Y) |
| ---- | --------------- | ------------ |
| <3.0 | Düşük           | No           |
| <3.0 | Yüksek          | Yes          |
| <3.0 | Yüksek          | Yes          |
| >3.0 | Düşük           | No           |
| >3.0 | Yüksek          | Yes          |
| >3.0 | Normal          | Yes          |
| <3.0 | Normal          | No           |
#### 4.1 İlk Stump ve Tahminler

Stump: İlk ağaç ve max_depth = 1 anlamına gelir.

- Interview Score tarafında entropy veya gini yapıldığına saflık en yüksek olan feature olduğunu görüyoruz. Çünkü Interviewda 3 Y/0 N varken GPA'da öyle bir durum yok.
- Dolayısıyla Interview Score Yüksek ise -> Yes, değil ise No

İlk Stump:

| ID        | GPA          | Interview Score | Approval (Y) | Tahmin     |
| --------- | ------------ | --------------- | ------------ | ---------- |
| 1         | <3.0         | Düşük           | No           | Yes        |
| 2         | <3.0         | Yüksek          | Yes          | Yes        |
| 3         | <3.0         | Yüksek          | Yes          | Yes        |
| 4         | >3.0         | Düşük           | No           | No         |
| 5         | >3.0         | Yüksek          | Yes          | Yes        |
| ~~**6**~~ | ~~**>3.0**~~ | ~~**Normal**~~  | ~~**Yes**~~  | ~~**No**~~ |
| 7         | <3.0         | Normal          | No           | Yes        |


![[Pasted image 20260520223106.png]]

Örneğin yukarıdaki fotoğrafta 6.index yanılıyor. Interview Score normal olup Approval Yes olmasına rağmen benim tahminim No olmuş.
Bu durumda bu satırla beraber istenilen kadar satırı alıp sonraki [[Decision Tree]]'ye aktarıyor ve böylelikle en optimum modeli bulmaya çalışıyor. Yani sadece hata değil, beraberinde train etmek için de datalar alıyor.

### 4.2 Hata Oranı ve Ağırlık Hesabı
#### Hesaplamalar
Başlangıçta tüm örnekler eşit ağırlıkta başlar. (Örneğin 7 tane row var ise 1/7 = 0.143)

Hata Oranı:
			Yanlış Sınıflanan Toplam Ağırlık
epsilon=	-------------------------------- = 0.143 / 1 = 0.143 
					Toplam Ağırlık

Örnekte sadece 6. satır yanlış hesaplandığından, yani 1 tane yanlış olduğundan Yanlış Sınıflanan Toplam Ağırlık 1 x 0.143 olarak hesaplandı. 2 tane olsaydı 2 x 0.143 olacaktı.

Not: Bu hata oranı ne kadar yüksek ise alfa o kadar düşük çıkar.

Ağırlık Katsayısı:

alfa = 1/2 In (1 - Hata Oranı (epsilon) /  Hata Oranı (epsilon))

- Bizim örnekte Hata Oranı kısımını ilgili yerlere koyduğumuzda ağırlık katsayısını 0.895 olarak buluyoruz. 
- Bu ağırlık oranı artık bu **decision tree'nin kullanacağı ağırlık katsayısıdır.**
- İlk stump'ın ağırlığı bulunduktan sonra, sonraki aşamalar için diğer row'ların da ağırlıkları tekrar güncellenir.

#### 4.3 Ağırlık Güncelleme Aşaması

Doğru sınıflananlar için:
wi' = wi (0.143 yani başlanan eşit ağırlık) X e ^ - Ağırlık Katsayısı

Örneğe göre ise: 0.143 X e^ - 0.895 = 0.058

Yanlış sınıflananlar için:
wi' = wi (0.143 yani başlanan eşit ağırlık) X e ^ + Ağırlık Katsayısı

Örneğe göre ise: 0.143 X e^ + 0.895 = 0.354

- Yani her bir satıra ağırlık verileceğinden, **eğer yanlış sınıflandırma yapılıyorsa ağırlığa fazla,** **doğru ise düşük verilerek modelin yanlış sınıflandırmalara ağırlık vermesi sağlanır.**
### 4.4 Normalizasyon
Yukarıda yapılan örnekte, hatalı olan 1 satırın ağırlık katsayısı 0.354 olarak diğerleri ise 0.56 olarak hesaplanmıştı. Normalizasyonda ise önce toplam ağırlıklar hesaplanır, daha sonrasında doğru ise doğru sınıfın katsayısı yanlış ise yanlış sınıfın katsayısı payda olacak şekilde bölünerek normalize edilen ağırlıklar oluşur.

Toplam: T = (6 X 0.058) + 0.354 = 0.702

Doğru sınıflananlar: 0.058 / 0.702 = 0.083
Yanlış sınıflananlar: 0.3534 / 0.702 = 0.504

Böylelike normalize edilen ağırlıklar oluşturularak bin aralıkları oluşturulur.

Normalizasyonun amacı: Ağırlıkları olasılık dağılımı gibi tutmaktır. Yani bunların ağırlıkların toplamı 1 olacak şekilde hale getirmek: 0.083 x 6 + 0.504 = 1.002
### 4.5 Bin Ataması

| ID  | Normalleştirilmiş Ağırlık | Bin Aralığı   |
| --- | ------------------------- | ------------- |
| 1   | 0.083                     | 0.000 - 0.083 |
| 2   | 0.083                     | 0.083 - 0.166 |
| 3   | 0.083                     | 0.166 - 0.249 |
| 4   | 0.083                     | 0.249 - 0.332 |
| 5   | 0.083                     | 0.332 - 0.415 |
| 6   | 0.504                     | 0.415 - 0.919 |
| 7   | 0.083                     | 0.919 - 1.000 |

Bu atamaların yapılmasının sebebi aşağıdaki konuda:

### 4.6 İkinci Stump ve Tahminler

Şimdi güncellenmiş veri ile ikinci stump seçilecek. Bu stump örneğin GPA > 3.0 mı? olsun diyelim. Evet ise Yes, Hayır ise No dönsün.

Bu seçilme esnasında, 0-1 arasında rastgele sayıların 6. satırdaki 0.415 - 0.919 aralığına düşme ihtimali çok daha yüksektir. Yani muhtemelen 7 değer seçilse 3-4 tanesi daha önce yanlış sınıflandırdığımız sınıfa düşeceğinden, model ID 6 üzerinde daha çok yoğunlaşacaktır.

İkinci Stump:

| ID  | GPA  | Interview Score | Approval (Y) | Tahmin | Doğru mu? |
| --- | ---- | --------------- | ------------ | ------ | --------- |
| 1   | <3.0 | Düşük           | No           | Yes    | Yes       |
| 2   | <3.0 | Yüksek          | Yes          | Yes    | No        |
| 3   | <3.0 | Yüksek          | Yes          | Yes    | No        |
| 4   | >3.0 | Düşük           | No           | No     | No        |
| 5   | >3.0 | Normal          | Yes          | Yes    | Yes       |
| 6   | >3.0 | Normal          | Yes          | No     | Yes       |
| 7   | >3.0 | Normal          | No           | Yes    | Yes       |
Bizim ilk stumpta bilemediğimiz: Interview Normal olup --> Yes verilen satırlar Bin aralığı yüksek olduğundan dolayı daha fazla gelmiş. (5-6-7. ID) Ve ilk stumpta bilemediğimiz satırların tamamını doğru bilmiş.
Ancak 2-3 ve 4 ID'leri yanlış sınıflandığından ayrı bir decision tree modeline giderek tekrar sınıflandırma yapılacak.

### Nihai Tahmin Fonksiyonu
AdaBoost nihai tahmin:

F(x) = a1 x h1(x) + a2 x h2(x)

ax --> Stump'ın ağırlık katsayısı

a1 = 0.895 (Interview score stump)
a2 = 0.552 (GPA Stump)
hm(x) = +1(YES), -1 (NO)

### Örnek Yeni Test Verisi
Test Verisi: 3.2
Interview Score = Yüksek

Yani her ikisinden de 1 aldığından katsayıların toplamı = 1.447 dolayısıyla sonuç 1 yani YES
### Adaboost ile Regresyon
#### Regresyon Temel Fikir
Adaboost Regressor'ın amacı:

![[Pasted image 20260523184228.png]]

Burada amaç bir regresyon değeri üretmek. Burada Y gerçek - Y Tahmin yapılarak örneğin hata büyüklüğü ölçülür. Kötü tahmin edilen gözlemlerin sample weightini arttırır.

#### Adım 1: İlk Model ve Residual Hesabı

| ID  | Experience | Salary |
| --- | ---------- | ------ |
| 1   | 1          | 40     |
| 2   | 2          | 45     |
| 3   | 3          | 50     |
| 4   | 4          | 58     |
| 5   | 5          | 60     |
| 6   | 6          | 65     |
Örnek veri setimiz yukarıdaki gibi. İlk olarak bir residual hesabı yapılacak. İlk model genellikle basit bir karar ağacı veya sabit bir tahmin olabilir. Örneğin:
F0(X) = Ortalama Salary = 53

| Gerçek Y | Tahmin F0(x) | Residual (r1 = Y - F0(x)) |
| -------- | ------------ | ------------------------- |
| 40       | 53           | 40-53 = -13               |
| 45       | 53           | 45-53 = -8                |
| 50       | 53           | 50-53 = -3                |
| 58       | 53           | 58-53 = 5                 |
| 60       | 53           | 60-53 = 7                 |
| 65       | 53           | 65-53 = 12                |
Sadece ortalama alıp hesaplama yapılan residuallar. Herhangi bir ağırlıklandırma yapılmadı.

#### Adım 2: Residual'lara Uygun Yeni Model
Yeni model h1(x), resiudal değerlerini tahmin etmeye çalışır. Basit bir karar ağacı, residual'lara göre bölünerek aşağıdaki gibi olabilir:

![[Pasted image 20260523185604.png]]

Yani ortalama 53 ise, 3 yıldan küçük ise ortalama 53-10'dan 43 dolar yapıyor. Yani residuala uygun bir model oluşturmuş olduk.

#### Adım 3: Yeni Residual ve İtersyon Devamı
Amaç, artıkların sıfıra yakınlaşmasıdır. Yani model artık hataları öğrenmeyi bırakana kadar eğitim devam eder.

### Sonuç:
Adaboost Regressor:
- Zayıf regresyon modellerinin artık hatalarını öğrenerek güçlü hale gelir.
- Her adımda residual'ları hedef olarak alır.
- Sınıflandırmaya göre yumuşak ve sayısal çıktılar üretir.


